In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from predictor.fetch import HRRRSource
from predictor.rules import (
    RuleBasedPredictor,
    MidHighCloudPresence, LowCloudObstruction,
    SolarAngleAtSunset, HumidityFactor,
)

In [ ]:
# --- Config: edit me ---
QUERY_TIME = datetime.now(timezone.utc) + timedelta(hours=1)  # next hour, UTC
BBOX = (-125.0, 25.0, -67.0, 49.0)  # (lon_min, lat_min, lon_max, lat_max) — CONUS
GRID_RES = 3.0  # degrees between sample points. Smaller = finer map, but each
                # grid point triggers a Herbie re-parse — 1.5 was too slow for
                # this MVP; revisit with in-memory caching in Phase 2.

In [ ]:
source = HRRRSource(cache_dir=Path("../../research/data/cache/hrrr"))
predictor = RuleBasedPredictor(
    rules=[
        MidHighCloudPresence(),
        LowCloudObstruction(),
        SolarAngleAtSunset(),
        HumidityFactor(),
    ],
    weights={
        "mid_high_cloud_presence": 2.0,
        "low_cloud_obstruction": 2.0,
        "solar_angle": 1.5,
        "humidity": 1.0,
    },
    source=source,
)

In [ ]:
lon_min, lat_min, lon_max, lat_max = BBOX
lons = np.arange(lon_min, lon_max + GRID_RES, GRID_RES)
lats = np.arange(lat_min, lat_max + GRID_RES, GRID_RES)

LON, LAT = np.meshgrid(lons, lats)
PROB = np.full_like(LON, np.nan, dtype=float)

# Note: HRRRSource caches Herbie downloads, so the first cell is slow,
# subsequent cells fast. For a single run cycle, every grid point hits the
# same GRIB2 file.
for j, lat in enumerate(lats):
    for i, lon in enumerate(lons):
        try:
            forecast = predictor.score(lat=float(lat), lon=float(lon), time=QUERY_TIME)
            PROB[j, i] = forecast.probability
        except Exception as e:
            # Outside HRRR domain or grid mismatch → leave NaN
            PROB[j, i] = np.nan

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax = plt.axes(projection=ccrs.LambertConformal(central_longitude=-96))
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.STATES, linewidth=0.4)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

mesh = ax.pcolormesh(
    LON, LAT, PROB,
    transform=ccrs.PlateCarree(),
    cmap="magma", vmin=0, vmax=1, shading="auto",
)
cbar = plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.04, fraction=0.05)
cbar.set_label("Fire-cloud probability")

ax.set_title(f"Firecloud forecast — {QUERY_TIME.isoformat()}")
plt.show()

In [ ]:
flat = PROB.flatten()
valid = ~np.isnan(flat)
order = np.argsort(flat[valid])
ranked_lats = LAT.flatten()[valid][order]
ranked_lons = LON.flatten()[valid][order]
ranked_probs = flat[valid][order]

print("Bottom 3 cells:")
for k in range(3):
    print(f"  ({ranked_lats[k]:.2f}, {ranked_lons[k]:.2f}) \u2192 {ranked_probs[k]:.2f}")

print("\nTop 3 cells:")
for k in range(1, 4):
    print(f"  ({ranked_lats[-k]:.2f}, {ranked_lons[-k]:.2f}) \u2192 {ranked_probs[-k]:.2f}")

# Detailed explanation for the top point
top_lat, top_lon = float(ranked_lats[-1]), float(ranked_lons[-1])
top_forecast = predictor.score(top_lat, top_lon, QUERY_TIME)
print("\nTop point detail:")
print(f"  {top_forecast.explanation}")
print(f"  inputs: {top_forecast.inputs}")